In [13]:
from pathlib import Path

import onnx

from hailo_sdk_client import ClientRunner

### Parsing ONNX to Hailo

Number of classes:
- cifar10: 10
- cifar100: 100
- flowers: 102
- food: 101
- pets: 37

In [14]:
chosen_hw_arch = 'hailo8'

# Create a classifier instance
device = "cpu"
batch_size = 8
num_classes = 102
channels = 3
img_h, img_w = 128, 128

optimization_level = 2  # -100, 0-4
compression_level = 0   # 0-5

model_name = 'resnet10t' 
model_frozen_layer_num = {
    'resnet18': 12,
    'resnet34': 20,
    'mobilenetv2_100': 21,
    'efficientnet_b2': 27,
    'visformer_tiny': 24,
    'lcnet_100': 19,
    'semnasnet_100': 20,
    'mobilenetv3_large_100': 22,
    'resnet10t': 14,
}
frozen_layer_num = model_frozen_layer_num[model_name]

artifacts_path = Path(f'../artifacts/{model_name}/frozen_layer_{frozen_layer_num}_classes_{num_classes}')

In [15]:
runner = ClientRunner(hw_arch=chosen_hw_arch)

model_path = artifacts_path / f'{model_name}.onnx'
start_node_names = ['input']
model_end_node_names = {
    'resnet18': ['/frozen_features/frozen_features.12/flatten/Flatten'],
    'resnet34': ['/frozen_features/frozen_features.20/flatten/Flatten'],
    'mobilenetv2_100': ['/frozen_features/frozen_features.21/flatten/Flatten'],
    'efficientnet_b2': ['/frozen_features/frozen_features.27/flatten/Flatten'],
    'visformer_tiny': [''],
    'lcnet_100': ['/frozen_features/frozen_features.19/Flatten'],
    'semnasnet_100': ['/frozen_features/frozen_features.20/flatten/Flatten'],
    'mobilenetv3_large_100': ['/frozen_features/frozen_features.22/Flatten'],
    'resnet10t': ['/frozen_features/frozen_features.14/flatten/Flatten']
}
end_node_names = model_end_node_names[model_name]
net_input_shapes = {'input': [1, channels, img_h, img_w],}

In [16]:
hn, npz = runner.translate_onnx_model(
    str(model_path),
    model_name,
    start_node_names=start_node_names,
    end_node_names=end_node_names,
    net_input_shapes=net_input_shapes,
)

[info] Translation started on ONNX model resnet10t
[info] Restored ONNX model resnet10t (completion time: 00:00:00.03)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:00.26)
[info] Start nodes mapped from original model: 'input': 'resnet10t/input_layer1'.
[info] End nodes mapped from original model: '/frozen_features/frozen_features.14/flatten/Flatten'.
[info] Translation completed on ONNX model resnet10t (completion time: 00:00:00.34)


In [17]:
hailo_har_model_path = artifacts_path / f'hailo8_optim_{optimization_level}_comp_{compression_level}' / f'{model_name}.har'
hailo_har_model_path.parent.mkdir(parents=True, exist_ok=True)
runner.save_har(hailo_har_model_path)

[info] Saved HAR to: /home/mateusz/Research/accelerator-training/artifacts/resnet10t/frozen_layer_14_classes_102/hailo8_optim_2_comp_0/resnet10t.har


In [18]:
# !hailo profiler {hailo_har_model_path}

#### Model optimization

In [19]:
import numpy as np
from tqdm import tqdm
from PIL import Image


images_list = list(Path('../data/ILSVRC2012_img_val/').rglob('*.JPEG'))

img_mean = [123.675, 116.28, 103.53]
img_std = [58.395, 57.12, 57.375]

# calib_dataset = np.random.randint(0, 255, (1024, img_h, img_w, channels))
calib_dataset = np.random.random((1024, img_h, img_w, channels))

## calib_dataset = np.zeros((len(images_list), img_h, img_w, channels))
# calib_set_size = 1024
# calib_dataset = np.zeros((calib_set_size, img_h, img_w, channels))
# for idx, img_path in enumerate(tqdm(sorted(images_list[:calib_set_size]))):
#     img = np.array(Image.open(img_path).convert('RGB').resize((img_h, img_w)))
#     img = (img.astype(np.float32) - img_mean) / img_std
#     # img = np.transpose(img, (2, 0, 1))
#     img = np.expand_dims(img, axis=0).astype(np.float32)
#     calib_dataset[idx, :, :, :] = img

# calib_set_path = Path('./calib_set.npy')
# np.save(calib_set_path, calib_dataset)

# calib_dataset = np.load(calib_set_path)
print(f'Calibration dataset shape: {calib_dataset.shape}')
print(f'Calibration dataset size: {calib_dataset.nbytes / 1024**2:.2f} MB')

Calibration dataset shape: (1024, 128, 128, 3)
Calibration dataset size: 384.00 MB


In [20]:
# Call Optimize to perform the optimization process
runner = ClientRunner(hw_arch=chosen_hw_arch, har=str(hailo_har_model_path))

# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)

# Batch size is 8 by default
# alls = 'normalization1 = normalization([123.675, 116.28, 103.53], [58.395, 57.12, 57.375])\n'

model_script_commands = [
    f'model_optimization_flavor(optimization_level={optimization_level}, compression_level={compression_level}, batch_size=8)\n'
]

# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(''.join(model_script_commands))

runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_har_model_path = str(hailo_har_model_path).replace('.har', '_quantized_model.har')
runner.save_har(quantized_har_model_path)

[info] Loading model script commands to resnet10t from string
[info] Found model with 3 input channels, using real RGB images for calibration instead of sampling random data.
[info] Starting Model Optimization


I0000 00:00:1755373349.825754  835684 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


[info] Model received quantization params from the hn


I0000 00:00:1755373350.125162  835684 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1755373350.535194  835684 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1755373350.847717  835684 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


[info] MatmulDecompose skipped
[info] Starting Mixed Precision
[info] Model Optimization Algorithm Mixed Precision is done (completion time is 00:00:00.28)
[info] LayerNorm Decomposition skipped
[info] Starting Statistics Collector
[info] Using dataset with 64 entries for calibration


...

128/128 ━━━━━━━━━━━━━━━━━━━━ 51s 78ms/step - _distill_loss_resnet10t/avgpool4: 0.0871 - total_distill_loss: 0.0871
Epoch 2/4
128/128 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - _distill_loss_resnet10t/avgpool4: 0.1197 - total_distill_loss: 0.1197
Epoch 3/4
128/128 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - _distill_loss_resnet10t/avgpool4: 0.0914 - total_distill_loss: 0.0914
Epoch 4/4
128/128 ━━━━━━━━━━━━━━━━━━━━ 10s 78ms/step - _distill_loss_resnet10t/avgpool4: 0.0724 - total_distill_loss: 0.0724
[info] Model Optimization Algorithm Quantization-Aware Fine-Tuning is done (completion time is 00:01:21.25)


I0000 00:00:1755373448.297415  835684 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1755373448.820463  835684 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


[info] Starting Layer Noise Analysis


Full Quant Analysis: 100%|██████████| 2/2 [00:12<00:00,  6.05s/iterations]


[info] Model Optimization Algorithm Layer Noise Analysis is done (completion time is 00:00:12.83)
[warning] The measured distribution for layer resnet10t/conv1 is different than expected. This could happened when the calibration set isn't normalized.
To disable this warning add the following line to the model script:
model_optimization_config(checker_cfg, batch_norm_checker=False)
[info] Output layers signal-to-noise ratio (SNR): measures the quantization noise (higher is better)
[info] 	resnet10t/output_layer1 SNR:	26.64 dB
[info] The calibration set seems to not be normalized, because the values range is [(8.5993355e-07, 0.99999994), (7.4357774e-07, 0.99999976), (7.246914e-08, 0.999998)].
Since the neural core works in 8-bit (between 0 to 255), a quantization will occur on the CPU of the runtime platform.
Add a normalization layer to the model to offload the normalization to the neural core.
Refer to the user guide Hailo Dataflow Compiler user guide / Model Optimization / Optimizatio

In [21]:
# runner.analyze_noise(calib_dataset, data_count=16, batch_size=2)

In [22]:
# !hailo profiler {quantized_har_model_path}

#### Quantized model compilation to HEF format

In [23]:
runner = ClientRunner(har=quantized_har_model_path)
hef = runner.compile()

hef_model_path = quantized_har_model_path.replace('har', 'hef')
with open(hef_model_path, 'wb') as f:
    f.write(hef)

[info] To achieve optimal performance, set the compiler_optimization_level to "max" by adding performance_param(compiler_optimization_level=max) to the model script. Note that this may increase compilation time.
[info] Loading network parameters
[info] Starting Hailo allocation and compilation flow
[info] Building optimization options for network layers...
[info] Successfully built optimization options - 1s 363ms
[info] Trying to compile the network in a single context
[info] Using Single-context flow
[info] Resources optimization params: max_control_utilization=75%, max_compute_utilization=75%, max_compute_16bit_utilization=75%, max_memory_utilization (weights)=75%, max_input_aligner_utilization=75%, max_apu_utilization=75%
[info] Validating layers feasibility
[info] output_layer1: Pass
[info] input_layer1: Pass
[info] conv1_sd1: Pass
[info] smuffers_shortcut_conv2_to_conv3: Pass
[info] conv3_sd2: Pass
[info] conv2_sd1: Pass
[info] conv1_sd0: Pass
[info] conv1_sdc: Pass
[info] sh_from

#### Export to ONNX with HailoOp operator

In [24]:
onnx_model = runner.get_hailo_runtime_model()
onnx_artifacts_path = hef_model_path.replace('.hef', '_hailo.onnx')
onnx_file = onnx.save(onnx_model, onnx_artifacts_path)
print(f'ONNX with HailoOp saved as: {onnx_artifacts_path}')

[warning] GlobalAvgPool output shape might be different from ONNX shape
ONNX with HailoOp saved as: ../artifacts/resnet10t/frozen_layer_14_classes_102/hailo8_optim_2_comp_0/resnet10t_quantized_model_hailo.onnx
